In [1]:
import os, sys, re, json, hashlib
from pathlib import Path
from datetime import datetime

NORTHSTAR = "agents-t14-qa"
NOTEBOOK_PATH = "notebooks/qa/44-agents-t14-qa.ipynb"
TOWER = 14
TOWER_NAME = "Tower of Agents"
MIN_SCORE = 70

BROWSER_ROOT = Path("/home/phuc/projects/solace-browser")
SRC = BROWSER_ROOT / "src"
WEB = BROWSER_ROOT / "web"
TESTS = BROWSER_ROOT / "tests"

results = []

def record(probe_id, name, passed, detail=""):
    status = "PASS" if passed else "FAIL"
    results.append({"id": probe_id, "name": name, "status": status, "detail": detail})
    print(f"{status}: {name}" + (f" -- {detail}" if detail else ""))

def read_file(path):
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.exists() else ""

print(f"Tower {TOWER}: {TOWER_NAME}")
assert BROWSER_ROOT.exists()

Tower 14: Tower of Agents


In [2]:
# Cell 1: OAuth3 Scope Enforcement (Floors 1-7)

scopes_py = read_file(SRC / "oauth3" / "scopes.py")
enforcement_py = read_file(SRC / "oauth3" / "enforcement.py")

# P1: Scope registry exists with platform.action.resource format
has_registry = 'SCOPE_REGISTRY' in scopes_py and 'platform' in scopes_py
record("P1", "OAuth3 SCOPE_REGISTRY exists with structured format", has_registry,
       "Triple-segment scope format (platform.action.resource)")

# P2: High-risk scopes identified
has_high_risk = 'HIGH_RISK_SCOPES' in scopes_py and 'risk_level' in scopes_py
record("P2", "HIGH_RISK_SCOPES set defined", has_high_risk,
       "High-risk scopes require step-up auth")

# P3: Destructive scopes identified
has_destructive = 'DESTRUCTIVE_SCOPES' in scopes_py and 'destructive' in scopes_py
record("P3", "DESTRUCTIVE_SCOPES set defined", has_destructive,
       "Destructive (irreversible) scopes tracked")

# P4: Scope validation function exists
has_validate = 'def validate_scopes' in scopes_py
record("P4", "validate_scopes() function exists", has_validate,
       "Scopes validated before use")

PASS: OAuth3 SCOPE_REGISTRY exists with structured format -- Triple-segment scope format (platform.action.resource)
PASS: HIGH_RISK_SCOPES set defined -- High-risk scopes require step-up auth
PASS: DESTRUCTIVE_SCOPES set defined -- Destructive (irreversible) scopes tracked
PASS: validate_scopes() function exists -- Scopes validated before use


In [3]:
# Cell 2: Four-Gate Enforcement (Floors 8-14)

# P5: ScopeGate class with 4 gates (G1-G4)
has_scope_gate = 'class ScopeGate' in enforcement_py
has_g1 = 'def g1_schema' in enforcement_py
has_g2 = 'def g2_expiry' in enforcement_py
has_g3 = 'def g3_scope' in enforcement_py
has_g4 = 'def g4_revocation' in enforcement_py
all_gates = has_scope_gate and has_g1 and has_g2 and has_g3 and has_g4
record("P5", "ScopeGate with all 4 gates (G1-G4)", all_gates,
       f"Gates: G1={has_g1}, G2={has_g2}, G3={has_g3}, G4={has_g4}")

# P6: Enforcement is fail-closed (GATE_BLOCKED default)
has_fail_closed = 'fail-closed' in enforcement_py.lower() and 'GATE_BLOCKED' in enforcement_py
record("P6", "Enforcement is fail-closed (deny by default)", has_fail_closed,
       "GATE_BLOCKED used as default rejection")

# P7: Unknown scopes treated as high risk
has_unknown_high = 'return "high"' in scopes_py  # get_scope_risk_level returns "high" for unknown
record("P7", "Unknown scopes default to high risk (fail-closed)", has_unknown_high,
       "get_scope_risk_level returns 'high' for unknown scopes")

PASS: ScopeGate with all 4 gates (G1-G4) -- Gates: G1=True, G2=True, G3=True, G4=True
PASS: Enforcement is fail-closed (deny by default) -- GATE_BLOCKED used as default rejection
PASS: Unknown scopes default to high risk (fail-closed) -- get_scope_risk_level returns 'high' for unknown scopes


In [4]:
# Cell 3: Step-Up Auth and Consent (Floors 15-21)

step_up_py = read_file(SRC / "oauth3" / "step_up.py")
consent_py = read_file(SRC / "oauth3" / "consent_ui.py")

# P8: Step-up nonce system exists
has_nonce = '_NONCE_STORE' in step_up_py or 'nonce' in step_up_py.lower()
record("P8", "Step-up nonce system exists", has_nonce,
       "One-time nonces for destructive action confirmation")

# P9: Step-up nonces are time-bounded (TTL)
has_ttl = 'TTL' in step_up_py or 'expires_at' in step_up_py
record("P9", "Step-up nonces have TTL (time-bounded)", has_ttl,
       "Nonces expire after TTL seconds")

# P10: Consent UI module exists
has_consent = len(consent_py) > 100
record("P10", "Consent UI module exists", has_consent,
       f"consent_ui.py: {len(consent_py)} chars")

# P11: Consent UI has grant/deny routes
has_consent_routes = '/consent' in consent_py or '/oauth3/consent' in consent_py
record("P11", "Consent UI has /consent route", has_consent_routes,
       "User can grant or deny scope requests")

PASS: Step-up nonce system exists -- One-time nonces for destructive action confirmation
PASS: Step-up nonces have TTL (time-bounded) -- Nonces expire after TTL seconds
PASS: Consent UI module exists -- consent_ui.py: 37469 chars
PASS: Consent UI has /consent route -- User can grant or deny scope requests


In [5]:
# Cell 4: Evidence Capture and Chain Integrity (Floors 22-28)

evidence_init = read_file(SRC / "evidence" / "__init__.py")

# P12: Evidence uses SHA-256 hash chain
has_sha256_chain = 'hashlib.sha256' in evidence_init and 'previous_hash' in evidence_init
record("P12", "Evidence uses SHA-256 hash chain", has_sha256_chain,
       "SHA-256 chaining with previous_hash links")

# P13: Evidence chain is verifiable (verify_chain function)
has_verify = 'def verify_chain' in evidence_init
record("P13", "Evidence chain has verify_chain() function", has_verify,
       "Full chain integrity verification supported")

# P14: Evidence uses deterministic JSON serialization
has_deterministic = 'sort_keys=True' in evidence_init
record("P14", "Evidence uses deterministic JSON (sort_keys=True)", has_deterministic,
       "Deterministic hashing via sorted JSON keys")

# P15: Evidence includes timestamps
has_timestamps = 'timestamp' in evidence_init and 'datetime' in evidence_init
record("P15", "Evidence entries include UTC timestamps", has_timestamps,
       "ISO 8601 UTC timestamps on all evidence")

PASS: Evidence uses SHA-256 hash chain -- SHA-256 chaining with previous_hash links
PASS: Evidence chain has verify_chain() function -- Full chain integrity verification supported
PASS: Evidence uses deterministic JSON (sort_keys=True) -- Deterministic hashing via sorted JSON keys
PASS: Evidence entries include UTC timestamps -- ISO 8601 UTC timestamps on all evidence


In [6]:
# Cell 5: Vault Security and Recipe Safety (Floors 29-35)

vault_py = read_file(SRC / "oauth3" / "vault.py")
recipe_exec = read_file(SRC / "recipes" / "recipe_executor.py")

# P16: Vault uses AES-256-GCM encryption
has_aes = 'AES-256-GCM' in vault_py and 'AESGCM' in vault_py
record("P16", "Vault uses AES-256-GCM encryption", has_aes,
       "Token storage encrypted with AES-256-GCM")

# P17: Vault key is 32 bytes (AES-256)
has_key_check = '32' in vault_py and ('encryption_key' in vault_py)
record("P17", "Vault enforces 32-byte encryption key", has_key_check,
       "AES-256 requires exactly 32-byte key")

# P18: Recipe executor maps actions to OAuth3 scopes
has_scope_map = 'ACTION_SCOPE_MAP' in recipe_exec
record("P18", "Recipe executor maps actions to OAuth3 scopes", has_scope_map,
       "Each recipe action is scope-gated")

# P19: Recipe executor uses evidence hashing
has_recipe_hash = 'hashlib' in recipe_exec or 'GENESIS_HASH' in recipe_exec
record("P19", "Recipe executor uses evidence hashing", has_recipe_hash,
       "Recipe execution creates hash-chained evidence")

PASS: Vault uses AES-256-GCM encryption -- Token storage encrypted with AES-256-GCM
PASS: Vault enforces 32-byte encryption key -- AES-256 requires exactly 32-byte key
PASS: Recipe executor maps actions to OAuth3 scopes -- Each recipe action is scope-gated
PASS: Recipe executor uses evidence hashing -- Recipe execution creates hash-chained evidence


In [7]:
# Cell 6: Agent Memory (YinYang) and Tool Registration (Floors 36-49)

# P20: YinYang JS files exist (agent state management)
yinyang_files = list((WEB / "js").glob("yinyang-*.js")) if (WEB / "js").exists() else []
record("P20", "YinYang JS modules exist (agent state)", len(yinyang_files) >= 3,
       f"Found {len(yinyang_files)} yinyang-*.js files: {[f.name for f in yinyang_files]}")

# P21: OAuth3 token revocation is supported
has_revoke = 'def revoke_token' in vault_py or 'revoke' in vault_py
record("P21", "Token revocation supported in vault", has_revoke,
       "Tokens can be immediately revoked")

# P22: Vault logs evidence events on token operations
has_evidence_log = 'evidence.log_event' in vault_py or 'log_event' in vault_py
record("P22", "Vault logs evidence events on token operations", has_evidence_log,
       "TOKEN_ISSUED, TOKEN_VALIDATED, TOKEN_REVOKED events logged")

# P23: Enforcement has step-up check for high-risk scopes
has_step_up_check = 'step_up' in enforcement_py.lower() and 'HIGH_RISK_SCOPES' in enforcement_py
record("P23", "Enforcement checks step-up for high-risk scopes", has_step_up_check,
       "High-risk actions require explicit re-consent")

# P24: Multiple platform scopes defined (linkedin, gmail, reddit, github)
platforms = set()
for scope_name in re.findall(r'"([a-z]+)\.[a-z]+\.[a-z]+"', scopes_py):
    platforms.add(scope_name.split('.')[0] if '.' in scope_name else scope_name)
# Also check the platform field
platform_fields = re.findall(r'"platform":\s*"([a-z]+)"', scopes_py)
unique_platforms = set(platform_fields)
record("P24", "Multiple platform scopes defined (>=5 platforms)", len(unique_platforms) >= 5,
       f"Platforms: {sorted(unique_platforms)}")

PASS: YinYang JS modules exist (agent state) -- Found 5 yinyang-*.js files: ['yinyang-rail.js', 'yinyang-tutorial-v2.js', 'yinyang-oauth3-confirm.js', 'yinyang-delight.js', 'yinyang-tutorial.js']
PASS: Token revocation supported in vault -- Tokens can be immediately revoked
PASS: Vault logs evidence events on token operations -- TOKEN_ISSUED, TOKEN_VALIDATED, TOKEN_REVOKED events logged
PASS: Enforcement checks step-up for high-risk scopes -- High-risk actions require explicit re-consent
PASS: Multiple platform scopes defined (>=5 platforms) -- Platforms: ['agent', 'browser', 'fs', 'github', 'gmail', 'hackernews', 'linkedin', 'machine', 'plugin', 'reddit']


In [8]:
passed = len([r for r in results if r["status"] == "PASS"])
failed = len([r for r in results if r["status"] == "FAIL"])
total = len(results)
score = round(passed / total * 100, 1) if total > 0 else 0
finding_rate = round(failed / total * 100, 1) if total > 0 else 0
grade = "A+" if score >= 95 else "A" if score >= 90 else "B+" if score >= 85 else "B" if score >= 80 else "C" if score >= 70 else "D" if score >= 60 else "F"
summary = {"surface": NORTHSTAR, "tower": TOWER, "tower_name": TOWER_NAME, "timestamp": datetime.now().isoformat(), "total_probes": total, "passed": passed, "failed": failed, "score": score, "grade": grade, "finding_rate": finding_rate, "converged": finding_rate < 5, "findings": [r for r in results if r["status"] == "FAIL"], "evidence_hash": hashlib.sha256(json.dumps(results, sort_keys=True).encode()).hexdigest()[:16]}
print("=" * 60)
print(f"TOWER {TOWER}: {TOWER_NAME}")
print("=" * 60)
print(f"Probes: {total} | Passed: {passed} | Failed: {failed}")
print(f"Score: {score}% | Grade: {grade} | Finding rate: {finding_rate}%")
print(f"Converged: {'YES' if summary['converged'] else 'NO'}")
print(f"Evidence hash: {summary['evidence_hash']}")
if summary["findings"]:
    print("\nFINDINGS:")
    for f in summary["findings"]:
        print(f"  [{f['id']}] {f['name']}: {f.get('detail', '')}")
print()
print(json.dumps(summary, indent=2))

TOWER 14: Tower of Agents
Probes: 24 | Passed: 24 | Failed: 0
Score: 100.0% | Grade: A+ | Finding rate: 0.0%
Converged: YES
Evidence hash: 88820420881704cb

{
  "surface": "agents-t14-qa",
  "tower": 14,
  "tower_name": "Tower of Agents",
  "timestamp": "2026-03-06T11:28:56.118717",
  "total_probes": 24,
  "passed": 24,
  "failed": 0,
  "score": 100.0,
  "grade": "A+",
  "finding_rate": 0.0,
  "converged": true,
  "findings": [],
  "evidence_hash": "88820420881704cb"
}
